In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings("ignore")
import shap
import scipy.stats as stats
import os
import torch
import torch.nn as nn

# 设置字体和绘图参数
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False

# 导入必要的基础类
from sklearn.base import BaseEstimator, RegressorMixin, clone

# 导入各种回归器
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, BayesianRidge, ARDRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, StackingRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn import metrics
from sklearn.multioutput import MultiOutputRegressor

# 导入其他回归模型
from xgboost import XGBRegressor
import xgboost as xgb
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from pytorch_tabnet.tab_model import TabNetRegressor
from joblib import dump
from sklearn.neighbors import KNeighborsRegressor
# from sklearn.ensemble import RandomForestRegressor
# 检查是否有 GPU 可用
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")

# 禁用 LightGBM 日志输出
os.environ["LIGHTGBM_SILENT"] = "true"  # 添加此行来禁用LightGBM警告

# 设置字体和绘图参数
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False
# TabNet包装器类，使其与sklearn API兼容
# TabNet包装器类，使其与sklearn API兼容
class TabNetRegressorWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, n_d=64, n_a=64, n_steps=4, gamma=1.0, lambda_sparse=0.0001, 
                 optimizer_params=dict(lr=0.01), batch_size=227, 
                 virtual_batch_size=96, max_epochs=100, patience=20, device_name='auto',
                 seed=42):
        self.n_d = n_d
        self.n_a = n_a
        self.n_steps = n_steps
        self.gamma = gamma
        self.lambda_sparse = lambda_sparse
        self.optimizer_params = optimizer_params
        self.batch_size = batch_size
        self.virtual_batch_size = virtual_batch_size
        self.max_epochs = max_epochs
        self.patience = patience
        self.device_name = device_name
        self.seed = seed
        self.model = None
        
    def fit(self, X, y):
        if len(y.shape) == 1:
            y = y.reshape(-1, 1)
            
        # 创建一个小的验证集用于早停
        from sklearn.model_selection import train_test_split
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.1, random_state=self.seed
        )
            
        self.model = TabNetRegressor(
            n_d=self.n_d,
            n_a=self.n_a,
            n_steps=self.n_steps,
            gamma=self.gamma,
            lambda_sparse=self.lambda_sparse,
            optimizer_params=self.optimizer_params,
            scheduler_params={"step_size": 50, "gamma": 0.9},
            mask_type='entmax',
            device_name=self.device_name,
            verbose=0,
            seed=self.seed
        )
        
        # 使用验证集进行早停，参考tablenet.ipynb中的实现
        self.model.fit(
            X_train=X_train,
            y_train=y_train,
            eval_set=[(X_val, y_val)],
            eval_name=['val'],
            eval_metric=['rmse'],
            max_epochs=self.max_epochs,
            patience=self.patience,
            batch_size=self.batch_size,
            virtual_batch_size=self.virtual_batch_size,
            num_workers=0
        )
        return self
    
    def predict(self, X):
        if self.model is None:
            raise ValueError("Model not fitted yet.")
        preds = self.model.predict(X)
        if preds.shape[1] == 1:
            return preds.ravel()
        return preds

# TabTransformer 实现
class TabTransformer(nn.Module):
    def __init__(self, input_dim, output_dim, embedding_dim=64, depth=1, 
                 heads=8, attn_dropout=0.225, ff_dropout=0.198):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        
        # 嵌入层 - 将输入特征转换为嵌入维度
        self.embedding = nn.Linear(input_dim, embedding_dim)
        
        # 位置编码 - 为每个特征添加位置信息
        self.pos_encoder = nn.Parameter(torch.zeros(1, input_dim, embedding_dim))
        
        # Transformer编码器层
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=heads,
            dim_feedforward=embedding_dim * 4,
            dropout=attn_dropout,
            activation='gelu',
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=depth)
        
        # MLP头部用于回归
        self.mlp_head = nn.Sequential(
            nn.Linear(embedding_dim * input_dim, embedding_dim * 2),
            nn.LayerNorm(embedding_dim * 2),
            nn.GELU(),
            nn.Dropout(ff_dropout),
            nn.Linear(embedding_dim * 2, embedding_dim),
            nn.LayerNorm(embedding_dim),
            nn.GELU(),
            nn.Dropout(ff_dropout),
            nn.Linear(embedding_dim, output_dim)
        )
        
    def forward(self, x):
        batch_size = x.size(0)
        
        # 重塑输入为(批大小, 特征数, 1)
        x = x.unsqueeze(-1)
        
        # 为每个特征创建嵌入
        x = x.repeat(1, 1, self.embedding.in_features)
        x = x * torch.eye(self.embedding.in_features, device=x.device)
        
        # 应用线性嵌入
        x = self.embedding(x)  # shape: [batch_size, num_features, embedding_dim]
        
        # 添加位置编码
        x = x + self.pos_encoder
        
        # 通过Transformer编码器
        x = self.transformer_encoder(x)
        
        # 重塑输出为(批大小, 特征数*嵌入维度)
        x = x.reshape(batch_size, -1)
        
        # 通过MLP头部进行回归预测
        x = self.mlp_head(x)
        
        return x

# TabTransformer包装器
class TabTransformerWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, input_dim=None, embedding_dim=64, depth=1, heads=8, 
                 attn_dropout=0.225, ff_dropout=0.198, learning_rate=0.00713,
                 batch_size=64, epochs=30, patience=10, device=None):
        self.input_dim = input_dim
        self.embedding_dim = embedding_dim
        self.depth = depth
        self.heads = heads
        self.attn_dropout = attn_dropout
        self.ff_dropout = ff_dropout
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.patience = patience
        self.device = device if device else ('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = None
        self.best_weights = None
        
    def fit(self, X, y):
        # 如果输入维度未指定，从数据推断
        if self.input_dim is None:
            self.input_dim = X.shape[1]
            
        # 确保y是二维的
        if len(y.shape) == 1:
            y = y.reshape(-1, 1)
            output_dim = 1
        else:
            output_dim = 1  # 这里强制为1，因为堆叠模型对每个输出单独训练
        
        # 转换为PyTorch张量
        X_torch = torch.tensor(X, dtype=torch.float32)
        y_torch = torch.tensor(y, dtype=torch.float32)
        
        # 创建数据集和数据加载器
        from torch.utils.data import TensorDataset, DataLoader, random_split
        dataset = TensorDataset(X_torch, y_torch)
        
        # 分割为训练集和验证集
        train_size = int(0.9 * len(dataset))
        val_size = len(dataset) - train_size
        train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
        
        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=self.batch_size)
        
        # 创建模型
        self.model = TabTransformer(
            input_dim=self.input_dim,
            output_dim=output_dim,
            embedding_dim=self.embedding_dim,
            depth=self.depth,
            heads=self.heads,
            attn_dropout=self.attn_dropout,
            ff_dropout=self.ff_dropout
        ).to(self.device)
        
        # 优化器和损失函数
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
        )
        loss_fn = nn.MSELoss()
        
        # 训练循环
        best_val_loss = float('inf')
        counter = 0
        
        for epoch in range(self.epochs):
            # 训练
            self.model.train()
            for batch_X, batch_y in train_loader:
                batch_X = batch_X.to(self.device)
                batch_y = batch_y.to(self.device)
                
                optimizer.zero_grad()
                pred = self.model(batch_X)
                loss = loss_fn(pred, batch_y)
                loss.backward()
                optimizer.step()
            
            # 验证
            self.model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch_X, batch_y in val_loader:
                    batch_X = batch_X.to(self.device)
                    batch_y = batch_y.to(self.device)
                    
                    pred = self.model(batch_X)
                    val_loss += loss_fn(pred, batch_y).item()
            
            val_loss /= len(val_loader)
            scheduler.step(val_loss)
            
            # 早停
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                counter = 0
                # 保存最佳模型权重
                self.best_weights = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
            else:
                counter += 1
                if counter >= self.patience:
                    break
        
        # 恢复最佳权重
        if self.best_weights is not None:
            self.model.load_state_dict(self.best_weights)
            self.model = self.model.to(self.device)
        
        return self
    
    def predict(self, X):
        if self.model is None:
            raise ValueError("Model not fitted yet.")
        
        # 转换为PyTorch张量
        X_torch = torch.tensor(X, dtype=torch.float32).to(self.device)
        
        # 预测
        self.model.eval()
        with torch.no_grad():
            predictions = self.model(X_torch).cpu().numpy()
        
        return predictions.reshape(-1)

# MLP包装器类（为了统一接口）
class MLPWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, hidden_layer_sizes=(100, 50), activation='relu',
                 solver='adam', alpha=0.0001, batch_size='auto',
                 learning_rate='adaptive', learning_rate_init=0.001,
                 max_iter=1000, tol=1e-4, early_stopping=True,
                 validation_fraction=0.1, n_iter_no_change=10, random_state=42):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.activation = activation
        self.solver = solver
        self.alpha = alpha
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.learning_rate_init = learning_rate_init
        self.max_iter = max_iter
        self.tol = tol
        self.early_stopping = early_stopping
        self.validation_fraction = validation_fraction
        self.n_iter_no_change = n_iter_no_change
        self.random_state = random_state
        self.model = None
    
    def fit(self, X, y):
        self.model = MLPRegressor(
            hidden_layer_sizes=self.hidden_layer_sizes,
            activation=self.activation,
            solver=self.solver,
            alpha=self.alpha,
            batch_size=self.batch_size,
            learning_rate=self.learning_rate,
            learning_rate_init=self.learning_rate_init,
            max_iter=self.max_iter,
            tol=self.tol,
            early_stopping=self.early_stopping,
            validation_fraction=self.validation_fraction,
            n_iter_no_change=self.n_iter_no_change,
            random_state=self.random_state
        )
        self.model.fit(X, y)
        return self
    
    def predict(self, X):
        return self.model.predict(X)
# 自定义多输出堆叠回归器（不使用交叉验证）
class MultiOutputStackingRegressorNoCv:
    def __init__(self, estimators, final_estimator):
        self.estimators = estimators
        self.final_estimator = final_estimator
        self.base_models = {}  # 存储基础模型
        self.meta_models = {}  # 存储元模型
    
    def fit(self, X, Y):
        n_targets = Y.shape[1]
        n_samples = X.shape[0]
        n_estimators = len(self.estimators)
        
        # 存储所有基础模型的预测结果
        meta_features = np.zeros((n_samples, n_estimators, n_targets))
        
        # 第一阶段：训练所有基础模型并获取它们对训练数据的预测
        for target_idx in range(n_targets):
            self.base_models[target_idx] = []
            y_target = Y[:, target_idx]
            
            for est_idx, (name, estimator) in enumerate(self.estimators):
                print(f"训练目标 {target_idx+1}/{n_targets} 的 {name} 模型...")
                # 克隆并训练基础模型
                model = clone(estimator)
                model.fit(X, y_target)
                self.base_models[target_idx].append(model)
                
                # 获取该模型对训练数据的预测
                meta_features[:, est_idx, target_idx] = model.predict(X)
        
        # 第二阶段：使用基础模型的预测作为特征来训练元模型
        for target_idx in range(n_targets):
            print(f"训练目标 {target_idx+1}/{n_targets} 的元模型...")
            # 提取当前目标的元特征（所有基础模型对该目标的预测）
            meta_features_target = meta_features[:, :, target_idx]
            y_target = Y[:, target_idx]
            
            # 克隆并训练元模型
            meta_model = clone(self.final_estimator)
            meta_model.fit(meta_features_target, y_target)
            self.meta_models[target_idx] = meta_model
        
        return self
    
    def predict(self, X):
        n_samples = X.shape[0]
        n_targets = len(self.meta_models)
        n_estimators = len(self.estimators)
        
        # 存储所有基础模型的预测
        meta_features = np.zeros((n_samples, n_estimators, n_targets))
        
        # 第一步：对每个目标使用对应的基础模型进行预测
        for target_idx in range(n_targets):
            for est_idx, model in enumerate(self.base_models[target_idx]):
                meta_features[:, est_idx, target_idx] = model.predict(X)
        
        # 第二步：使用元模型对基础模型的预测结果进行集成预测
        y_pred = np.zeros((n_samples, n_targets))
        for target_idx, meta_model in self.meta_models.items():
            meta_features_target = meta_features[:, :, target_idx]
            y_pred[:, target_idx] = meta_model.predict(meta_features_target)
        
        return y_pred

    def transform(self, X, target_idx=0):
        """
        将输入数据转换为基础模型的预测结果
        用于生成二级学习器的输入特征
        """
        if target_idx not in self.meta_models:
            raise ValueError(f"Target index {target_idx} not found in trained models")
            
        n_samples = X.shape[0]
        n_estimators = len(self.estimators)
        
        # 存储指定目标的基础模型预测
        meta_features = np.zeros((n_samples, n_estimators))
        
        # 获取每个基础模型对指定目标的预测
        for est_idx, model in enumerate(self.base_models[target_idx]):
            meta_features[:, est_idx] = model.predict(X)
            
        return meta_features
    
    def get_meta_model(self, target_idx=0):
        """
        获取指定目标的二级学习器
        """
        if target_idx not in self.meta_models:
            raise ValueError(f"Target index {target_idx} not found in trained models")
            
        return self.meta_models[target_idx]
    
    def get_base_models(self, target_idx=0):
        """
        获取指定目标的所有基础学习器
        """
        if target_idx not in self.base_models:
            raise ValueError(f"Target index {target_idx} not found in trained models")
            
        return self.base_models[target_idx]

# 导入CTGAN增强数据
df_augmented = pd.read_excel('data/CTGAN_data_multi.xlsx')
print(f"增强数据集大小: {df_augmented.shape}")

# 导入原始数据用于测试评估
df_original = pd.read_excel('data/new_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 提取多个目标变量
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 从增强数据中分离特征和目标
X_augmented = df_augmented.drop(target_vars, axis=1)
y_augmented = df_augmented[target_vars]

# 从原始数据中分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 将原始数据分为训练集和测试集
X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(
    X_original, y_original, test_size=0.2, random_state=42
)

# 创建标准化器
scaler_X = StandardScaler()
scaler_y = StandardScaler()

# 对增强数据进行标准化
X_augmented_scaled = scaler_X.fit_transform(X_augmented)
y_augmented_scaled = scaler_y.fit_transform(y_augmented)

# 对原始测试数据进行标准化
X_test_scaled = scaler_X.transform(X_test_orig)
y_test_scaled = scaler_y.transform(y_test_orig)

# 定义基础学习器
base_learners = [
    ("RF", RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_split=4,
        min_samples_leaf=3,
        max_features=None,
        n_jobs=-1,
        random_state=42
    )),
    
    ("LGBM", LGBMRegressor(
        boosting_type='gbdt',
        objective='regression',
        metric='rmse',
        num_leaves=31,
        max_depth=13,
        learning_rate=0.05,
        feature_fraction=1.0,
        subsample=0.52,
        min_child_samples=4,
        subsample_freq=5,
        verbose=-1,
        n_jobs=-1,
        n_estimators=308,
        random_state=42
    )),
    
    ("XGB", XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        n_jobs=-1,
        random_state=42
    )),
    
    ("ET", ExtraTreesRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=2,
        min_samples_leaf=3,
        max_features='auto',
        bootstrap=False,
        n_jobs=-1,
        random_state=42
    )),
    
    # 添加TabTransformer作为第五个基础学习器
    ("TabTransformer", TabTransformerWrapper(
        input_dim=X_augmented_scaled.shape[1],  # 特征维度
        embedding_dim=64,         # 优化参数
        depth=1,                  # 优化参数
        heads=8,                  # 优化参数
        attn_dropout=0.225,       # 优化参数
        ff_dropout=0.198,         # 优化参数
        learning_rate=0.00713,    # 优化参数
        batch_size=64,            # 优化参数
        epochs=30,                # 简化训练
        patience=10,              # 早停参数
        device=device             # 使用GPU
    )),
    
    # 添加MLP作为第六个基础学习器
    ("MLP", MLPWrapper(
        hidden_layer_sizes=(100, 50),  # 两个隐藏层
        activation='relu',            
        solver='adam',              
        alpha=0.0001,               
        batch_size='auto',          
        learning_rate='adaptive',   
        learning_rate_init=0.001,   
        max_iter=1000,              
        tol=1e-4,                   
        early_stopping=True,        
        validation_fraction=0.1,    
        n_iter_no_change=10,        
        random_state=42
    )),
    
        # 添加TabNet作为第七个基础学习器
    ("TabNet", TabNetRegressorWrapper(
        n_d=64,                    # 参考tablenet.ipynb中的配置
        n_a=64,                    # 参考tablenet.ipynb中的配置
        n_steps=4,                 # 参考tablenet.ipynb中的配置
        gamma=1.0,                 # 保持不变
        lambda_sparse=0.0001,      # 保持不变
        optimizer_params=dict(lr=0.005),  # 参考tablenet.ipynb中的学习率
        batch_size=227,            # 保持不变
        virtual_batch_size=96,     # 保持不变
        max_epochs=200,            # 参考tablenet.ipynb中设置为200
        patience=20,               # 保持不变
        device_name=device,        # 使用GPU
        seed=42
    )),("SVR",SVR(
    kernel='rbf',  # 径向基函数核
    C=0.11552292675414941,         # 正则化参数
    epsilon=0.01,   # SVR中的epsilon参数
    gamma=0.26208814362264377, # 核系数
    cache_size=1000 # 缓存大小，加速训练
)),("KNN",KNeighborsRegressor(
    n_neighbors=2,  # 邻居数量
    weights='uniform',  # 权重类型：'uniform'或'distance'
    algorithm='auto',  # 算法：'auto', 'ball_tree', 'kd_tree', 'brute'
    p=9,  # 距离度量，2表示欧氏距离
    n_jobs=-1  # 使用所有可用的CPU核心
))
]
# 定义二级学习器
meta_model = LinearRegression()

# 创建多输出堆叠回归器
stacking_regressor = MultiOutputStackingRegressorNoCv(
    estimators=base_learners, 
    final_estimator=meta_model
)

# 训练模型（使用增强后的数据）
print("开始训练模型...")
stacking_regressor.fit(X_augmented_scaled, y_augmented_scaled)
print("模型训练完成！")

# 使用训练好的模型在测试集上预测
y_pred_test_scaled = stacking_regressor.predict(X_test_scaled)

# 将预测结果反标准化回原始空间
y_pred_test = scaler_y.inverse_transform(y_pred_test_scaled)

# 真实值
y_test_true = y_test_orig.values

# 为每个输出目标计算评估指标
print("评估指标:")
for idx, name in enumerate(target_vars):
    # 测试集指标
    mse_test = metrics.mean_squared_error(y_test_true[:, idx], y_pred_test[:, idx])
    rmse_test = np.sqrt(mse_test)
    mae_test = metrics.mean_absolute_error(y_test_true[:, idx], y_pred_test[:, idx])
    r2_test = metrics.r2_score(y_test_true[:, idx], y_pred_test[:, idx])
   
    print(f"\n{name} 指标:")
    print(f"测试: MSE={mse_test:.4f}, RMSE={rmse_test:.4f}, MAE={mae_test:.4f}, R²={r2_test:.4f}")


In [ ]:
import seaborn as sns
# 添加到现有代码结尾处，SHAP分析完成后

# 创建一个函数来标准化SHAP值
def normalize_shap(shap_values):
    """将SHAP值标准化到[-1, 1]范围内"""
    abs_max = np.abs(shap_values).max()
    if abs_max > 0:
        return shap_values / abs_max
    return shap_values

# 存储所有目标的元模型SHAP值
all_meta_shap_values = []
all_meta_features = []
base_model_names = [name for name, _ in base_learners]

# 为每个目标变量计算并收集SHAP值
print("\n计算所有目标变量的归一化SHAP值汇总...")
for target_idx, target_name in enumerate(target_vars):
    try:
        # 获取这个目标的元模型
        meta_model = stacking_regressor.get_meta_model(target_idx)
        
        # 获取一级学习器的预测结果
        meta_features = stacking_regressor.transform(X_test_scaled, target_idx)
        all_meta_features.append(meta_features)
        
        # 使用KernelExplainer分析元模型
        meta_explainer = shap.KernelExplainer(meta_model.predict, meta_features)
        meta_shap_values = meta_explainer.shap_values(meta_features)
        
        # 标准化SHAP值
        normalized_shap = normalize_shap(meta_shap_values)
        all_meta_shap_values.append(normalized_shap)
        
        print(f"  已处理目标 {target_name} 的SHAP值")
        
    except Exception as e:
        print(f"  处理目标 {target_name} 时出错: {str(e)}")

# 确保我们至少有一些SHAP值可用
if all_meta_shap_values:
    # 计算归一化后的平均SHAP值
    avg_meta_features = np.mean(all_meta_features, axis=0)
    avg_meta_shap = np.mean(all_meta_shap_values, axis=0)
    
    # 创建汇总可视化
    plt.figure(figsize=(12, 8), dpi=300)
    
    # 点图 - 显示所有基础学习器的综合贡献
    shap.summary_plot(
        avg_meta_shap,
        avg_meta_features,
        feature_names=base_model_names,
        plot_type="dot",
        show=False,
        cmap=plt.cm.Purples_r  # 使用蓝绿色调
    )
    plt.title("Overall Meta Learner - Base Learner Contribution Analysis (All Target Variables)", fontsize=16)
    plt.tight_layout()
    plt.savefig("overall_meta_learner_contribution.pdf", format='pdf', bbox_inches='tight')
    plt.show()

print("\nNormalized SHAP value analysis completed!")


In [ ]:
# 检查当前 base_model_names 中是否包含 'TabTransformer'
if 'TabTransformer' in base_model_names:
    # 找到 'TabTransformer' 的索引
    index = base_model_names.index('TabTransformer')
    # 将其替换为带有换行符的版本
    base_model_names[index] = 'TabTra-\nnsformer'

In [ ]:
import shap

# 假设 X_test 是一个 Pandas DataFrame
feature_importance = np.mean(np.abs(avg_meta_shap), axis=0)

# 创建一个 DataFrame，将列名设置为 X_test 的特征名
feature_importance_df = pd.DataFrame({
    'Feature': base_model_names,  # 使用 X_test 的特征名
    'Importance': feature_importance  # 对应的均值
}).sort_values(by='Importance', ascending=False)  # 按重要性降序排序
feature_importance_df

In [ ]:
base_model_names

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import shap

# 设置字体为 Times New Roman
plt.rcParams.update({'font.family': 'Times New Roman'})

# 提取特征名和重要性值
features = feature_importance_df['Feature']
importances = feature_importance_df['Importance']

# 原始颜色列表
colors_original = ['#FDD379', '#F7C2CD', '#A6DAEF', '#B0D9A5', '#E68D3D', '#E26472', '#6270B7', '#077535']

# 新增颜色
colors_new = ['#8E44AD','#FFD700',  '#3498DB', '#16A085', '#D35400', '#C0392B', '#7F8C8D', '#2ECC71']

# 合并原始颜色和新增颜色
colors = colors_original + colors_new
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
# 确保颜色数量与特征数量一致（截取匹配数量）
colors = colors[:len(features)]

# 创建一行两列的子图布局
fig, axes = plt.subplots(1, 2, figsize=(16, 10), gridspec_kw={'width_ratios': [1.6, 1]})

# 创建一行两列的子图布局
fig, axes = plt.subplots(1, 2, figsize=(16, 10), gridspec_kw={'width_ratios': [1.6, 1]})
plt.sca(axes[0])  # 设置当前子图为左侧

# 绘制左侧 SHAP 蜂巢图

start_color = (228/255, 107/255, 144/255)  # 修正超出范围的值
end_color = (130/255, 170/255, 231/255)

# 创建渐变色映射
custom_cmap = LinearSegmentedColormap.from_list("custom_pink_blue", 
                                              [start_color, (1, 1, 1), end_color])


# # 设置起始和结束颜色
# start_color = (228/255, 107/255, 144/255)  # 粉红色
# end_color = (130/255, 170/255, 231/255)    # 蓝色

# # 计算一个偏紫色的中间色调（两种颜色的平均值）
# mid_color = ((start_color[0] + end_color[0])/2, 
#              (start_color[1] + end_color[1])/2,
#              (start_color[2] + end_color[2])/2)

# # 创建渐变色映射 - 使用计算出的中间色
# custom_cmap = LinearSegmentedColormap.from_list("custom_pink_blue", 
#                                               [start_color, mid_color, end_color],
#                                               N=256)



shap.summary_plot(avg_meta_shap, avg_meta_features, feature_names=base_model_names, plot_type="dot", show=False,cmap=custom_cmap)
# plt.cm.RdBu_r
axes[0].set_title('SHAP Summary Plot', fontsize=17)
# 调整图 a 的字体大小和布局
plt.tick_params(axis='x', labelsize=15)  # 增加x轴刻度标签大小
plt.tick_params(axis='y', labelsize=15)  # 增加y轴刻度标签大小
cbar = plt.gcf().axes[-1]  # 获取colorbar轴（假设是最后一个轴）
cbar.tick_params(labelsize=15)  # 调整colorbar刻度标签大小
cbar.set_ylabel("Feature Value", fontsize=15, labelpad=-3, family="Times New Roman")  # 设置colorbar标签的字体和移动位置
cbar.yaxis.set_label_coords(3, 0.5)  # 移动标签到左边
plt.xlabel("Attributions value", fontsize=15, family="Times New Roman")

# # 设置子图位置和大小 - 向左移动环形图
# # [left, bottom, width, height] - left值减小会使图向左移动
# axes[1].set_position([0.5, 0.1, 0.4, 0.8])  # 调整left值（第一个参数）使其更小
# 绘制右侧环形图
plt.sca(axes[1])  # 设置当前子图为右侧
wedges, texts = axes[1].pie(
    importances,
    startangle=90,
    radius=1.5,
    wedgeprops=dict(width=0.7, edgecolor='white', linewidth=2),  # 设置间隔
    colors=colors,  # 使用扩展后的颜色
    labels=None  # 不在区域中显示标签
)

# 添加百分比到区域外部
for i, (wedge, importance) in enumerate(zip(wedges, importances)):
    angle = (wedge.theta2 - wedge.theta1) / 2 + wedge.theta1  # 计算区域角度
    x = 1.2 * wedge.r * np.cos(np.radians(angle))  # 调整百分比位置 (x 坐标)
    y = 1.2 * wedge.r * np.sin(np.radians(angle))  # 调整百分比位置 (y 坐标)
    
    percentage = importance / sum(importances)
    
    if percentage < 0.03:  # 针对小于4.6%的值，不显示
        # 不显示任何内容
        pass
    elif percentage < 0.01:  # 针对小于1%但大于等于0.046%的值，用连线标注
        # 外部连接点
        x_annotate = 1 * wedge.r * np.cos(np.radians(angle))
        y_annotate = 1.4 * wedge.r * np.sin(np.radians(angle))
        # 绘制连线
        axes[1].annotate(
            f'{percentage:.1%}', 
            xy=(x, y), 
            xytext=(x_annotate, y_annotate),
            arrowprops=dict(arrowstyle="-", color='black'),
            ha='center', 
            va='center', 
            fontsize=14
        )
    else:
        # 正常显示百分比
        axes[1].text(x, y, f'{percentage:.1%}', ha='center', va='center', fontsize=14)

# 添加环形图中心文本
axes[1].text(0, 0, 'metal model', ha='center', va='center', fontsize=16)

# 添加图例
axes[1].legend(
    wedges, features,
    title="Features",
    loc="center left",
    bbox_to_anchor=(1.45, 0.5),  # 图例位置，调整到右侧
    fontsize=14,
    title_fontsize=14,
)

axes[1].set_title('Feature Importance Donut Chart', fontsize=18, 
                  pad=0,             # 增加 pad 值，控制标题与图之间的距离
                  y=1.445,             # y 位置：1.0 是默认位置，>1 表示上移，<1 表示下移
                  x=0.65)              # x 位置：0.5 是居中，<0.5 向左移，>0.5 向右移

# 调整布局并保存
# plt.tight_layout()
plt.savefig("7.pdf", format='pdf', bbox_inches='tight')
plt.show()
